# ML Results Analysis

Comprehensive analysis of the leave-one-dataset-out (LOO) ML experiments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load results
results_path = Path('../database/ml_results.parquet')
df = pd.read_parquet(results_path)

print(f"Results shape: {df.shape}")
print(f"\nColumns: {list(df.columns)[:20]}...")  # First 20 columns

## 1. Data Overview

In [ ]:
# Basic info
print("Dataset Info:")
print(f"Total rows: {len(df)}")
print(f"\nUnique models: {df['model_name'].nunique()}")
print(f"Unique test datasets: {df['test_dataset'].nunique()}")
print(f"Unique symbols: {df['symbol'].nunique()}")
print(f"\nModels: {sorted(df['model_name'].unique())}")
print(f"\nTest datasets:\n{sorted(df['test_dataset'].unique())}")

In [ ]:
# Data types and missing values
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nKey metrics dtypes:")
metrics_cols = ['test_roc_auc', 'test_accuracy', 'total_return_pct', 'sharpe', 'max_drawdown_pct']
print(df[metrics_cols].dtypes)

## 2. Model Performance: ROC-AUC and Accuracy

In [ ]:
# Group by model
model_perf = df.groupby('model_name')[['test_roc_auc', 'test_accuracy', 'cv_best_roc_auc']].agg(['mean', 'std', 'min', 'max'])
print("Model Performance Summary:")
print(model_perf.round(4))

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-AUC by model
df.boxplot(column='test_roc_auc', by='model_name', ax=axes[0])
axes[0].set_title('Test ROC-AUC by Model')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('ROC-AUC')
plt.sca(axes[0])
plt.xticks(rotation=45)

# Accuracy by model
df.boxplot(column='test_accuracy', by='model_name', ax=axes[1])
axes[1].set_title('Test Accuracy by Model')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Accuracy')
plt.sca(axes[1])
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print("\nMean Test ROC-AUC by Model:")
print(df.groupby('model_name')['test_roc_auc'].mean().sort_values(ascending=False))

## 3. Backtest Metrics Analysis

In [ ]:
# Backtest performance summary
backtest_cols = ['total_return_pct', 'buy_hold_return_pct', 'excess_return_pct', 'sharpe', 'max_drawdown_pct']
backtest_summary = df[backtest_cols].describe()
print("Backtest Metrics Summary:")
print(backtest_summary.round(4))

In [ ]:
# Returns comparison: Model strategy vs Buy-Hold
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df))
width = 0.35

ax.bar(x - width/2, df['total_return_pct'], width, label='Model Strategy', alpha=0.8)
ax.bar(x + width/2, df['buy_hold_return_pct'], width, label='Buy & Hold', alpha=0.8)

ax.set_xlabel('Test Run')
ax.set_ylabel('Return (%)')
ax.set_title('Model Strategy Returns vs Buy-Hold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Sharpe ratio distribution
fig, ax = plt.subplots(figsize=(12, 6))

df.boxplot(column='sharpe', by='model_name', ax=ax)
ax.set_title('Sharpe Ratio by Model')
ax.set_xlabel('Model')
ax.set_ylabel('Sharpe Ratio')
plt.sca(ax)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nSharpe Ratio Statistics by Model:")
print(df.groupby('model_name')['sharpe'].describe().round(4))

## 4. Performance by Dataset

In [ ]:
# Group by test dataset
dataset_perf = df.groupby('test_dataset')[['test_roc_auc', 'test_accuracy', 'total_return_pct', 'sharpe']].mean()
print("Average Performance by Test Dataset:")
print(dataset_perf.round(4).sort_values('test_roc_auc', ascending=False))

In [ ]:
# Visualize performance by dataset
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ROC-AUC by dataset
dataset_perf_sorted = dataset_perf.sort_values('test_roc_auc', ascending=False)
dataset_perf_sorted['test_roc_auc'].plot(kind='barh', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Average ROC-AUC by Dataset')
axes[0, 0].set_xlabel('ROC-AUC')

# Accuracy by dataset
dataset_perf_sorted = dataset_perf.sort_values('test_accuracy', ascending=False)
dataset_perf_sorted['test_accuracy'].plot(kind='barh', ax=axes[0, 1], color='seagreen')
axes[0, 1].set_title('Average Accuracy by Dataset')
axes[0, 1].set_xlabel('Accuracy')

# Return by dataset
dataset_perf_sorted = dataset_perf.sort_values('total_return_pct', ascending=False)
dataset_perf_sorted['total_return_pct'].plot(kind='barh', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Average Total Return by Dataset')
axes[1, 0].set_xlabel('Total Return (%)')

# Sharpe by dataset
dataset_perf_sorted = dataset_perf.sort_values('sharpe', ascending=False)
dataset_perf_sorted['sharpe'].plot(kind='barh', ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Average Sharpe Ratio by Dataset')
axes[1, 1].set_xlabel('Sharpe Ratio')

plt.tight_layout()
plt.show()

## 5. Best Performing Models

In [ ]:
# Top performers by ROC-AUC
print("Top 10 Best Performers (by ROC-AUC):")
top_roc = df.nlargest(10, 'test_roc_auc')[['test_dataset', 'model_name', 'test_roc_auc', 'test_accuracy', 'sharpe', 'total_return_pct']]
print(top_roc.to_string(index=False))

In [ ]:
# Top performers by Sharpe ratio
print("\nTop 10 Best Performers (by Sharpe Ratio):")
top_sharpe = df.nlargest(10, 'sharpe')[['test_dataset', 'model_name', 'sharpe', 'total_return_pct', 'max_drawdown_pct', 'test_roc_auc']]
print(top_sharpe.to_string(index=False))

In [ ]:
# Top performers by Total Return
print("\nTop 10 Best Performers (by Total Return):")
top_return = df.nlargest(10, 'total_return_pct')[['test_dataset', 'model_name', 'total_return_pct', 'sharpe', 'test_roc_auc', 'max_drawdown_pct']]
print(top_return.to_string(index=False))

## 6. Model vs Dataset Heatmap

In [ ]:
# Create pivot table for heatmap
pivot_roc = df.pivot_table(values='test_roc_auc', index='test_dataset', columns='model_name', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(pivot_roc, annot=True, fmt='.3f', cmap='RdYlGn', center=0.5, ax=ax, cbar_kws={'label': 'ROC-AUC'})
ax.set_title('ROC-AUC Heatmap: Models vs Datasets')
plt.tight_layout()
plt.show()

In [ ]:
# Return heatmap
pivot_return = df.pivot_table(values='total_return_pct', index='test_dataset', columns='model_name', aggfunc='mean')

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(pivot_return, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax, cbar_kws={'label': 'Return (%)'})
ax.set_title('Total Return Heatmap: Models vs Datasets')
plt.tight_layout()
plt.show()

## 7. Correlation Analysis

In [ ]:
# Correlation between metrics
metrics = ['test_roc_auc', 'test_accuracy', 'cv_best_roc_auc', 'total_return_pct', 'sharpe', 'max_drawdown_pct', 'annualized_return_pct']
corr_matrix = df[metrics].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Correlation Matrix: ML Metrics')
plt.tight_layout()
plt.show()

print("\nCorrelation with Total Return:")
print(corr_matrix['total_return_pct'].sort_values(ascending=False))

## 8. Summary Statistics

In [ ]:
# Overall summary
print("OVERALL EXPERIMENT SUMMARY")
print("="*60)
print(f"Total experiments: {len(df)}")
print(f"Datasets tested: {df['test_dataset'].nunique()}")
print(f"Models evaluated: {df['model_name'].nunique()}")
print(f"Average runs per dataset: {len(df) / df['test_dataset'].nunique():.1f}")

print("\nPREDICTIVE PERFORMANCE:")
print(f"  Mean ROC-AUC: {df['test_roc_auc'].mean():.4f} (+/- {df['test_roc_auc'].std():.4f})")
print(f"  Mean Accuracy: {df['test_accuracy'].mean():.4f} (+/- {df['test_accuracy'].std():.4f})")
print(f"  Mean CV ROC-AUC: {df['cv_best_roc_auc'].mean():.4f}")

print("\nBACKTEST PERFORMANCE:")
print(f"  Mean Total Return: {df['total_return_pct'].mean():.2f}%")
print(f"  Mean Buy-Hold Return: {df['buy_hold_return_pct'].mean():.2f}%")
print(f"  Mean Excess Return: {df['excess_return_pct'].mean():.2f}%")
print(f"  Mean Sharpe Ratio: {df['sharpe'].mean():.4f}")
print(f"  Mean Max Drawdown: {df['max_drawdown_pct'].mean():.2f}%")
print(f"  Mean Annualized Return: {df['annualized_return_pct'].mean():.2f}%")

print("\nTRADE STATISTICS:")
print(f"  Mean # of Trades: {df['num_trades'].mean():.1f}")
print(f"  Mean Signal Coverage: {df['signal_coverage'].mean():.2%}")
print(f"  Mean Train Rows: {df['n_train_rows'].mean():.0f}")
print(f"  Mean Test Rows: {df['n_test_rows'].mean():.0f}")

In [ ]:
# Models ranked by different criteria
print("\nMODEL RANKINGS:")
print("\nBy ROC-AUC (test):")
print(df.groupby('model_name')['test_roc_auc'].mean().sort_values(ascending=False).round(4))

print("\nBy Sharpe Ratio:")
print(df.groupby('model_name')['sharpe'].mean().sort_values(ascending=False).round(4))

print("\nBy Total Return:")
print(df.groupby('model_name')['total_return_pct'].mean().sort_values(ascending=False).round(2))

print("\nBy Excess Return (vs Buy-Hold):")
print(df.groupby('model_name')['excess_return_pct'].mean().sort_values(ascending=False).round(2))